# 🤖 Trading Bot - Entraînement Complet

## Pipeline d'Entraînement Professionnel

Ce notebook contient tout le nécessaire pour entraîner ton bot de trading:

1. **Setup** - Installation des dépendances
2. **Données** - Téléchargement economic calendar + chargement Gold data
3. **Entraînement** - Training avec curriculum learning
4. **Évaluation** - Métriques de performance
5. **Paper Trading** - Test en mode simulation

---
**⚠️ IMPORTANT:** Active le GPU dans Runtime → Change runtime type → GPU

## 1. Setup - Installation des Dépendances

In [ ]:
# Installation des packages requis
!pip install -q stable-baselines3[extra] gymnasium pandas numpy matplotlib seaborn
!pip install -q torch tensorboard scikit-learn ta fredapi requests
!pip install -q arch  # Pour GARCH

print("✅ Packages installés!")

In [ ]:
# Vérifier le GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Pas de GPU détecté - l'entraînement sera plus lent")

In [ ]:
# Cloner le repo (ou uploader tes fichiers)
# Option 1: Cloner depuis GitHub
# !git clone https://github.com/ton-username/TradingBOT_Agentic.git
# %cd TradingBOT_Agentic

# Option 2: Uploader depuis Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copier les fichiers depuis Drive (ajuste le chemin)
# !cp -r "/content/drive/MyDrive/TradingBOT_Agentic" /content/
# %cd /content/TradingBOT_Agentic

## 2. Téléchargement des Données Économiques

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Créer les dossiers
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)

In [ ]:
# =============================================================================
# ECONOMIC CALENDAR - VRAIES DONNÉES HISTORIQUES
# =============================================================================

# NFP Historical (en milliers de jobs)
NFP_DATA = {
    2019: [304, 56, 189, 263, 75, 224, 164, 130, 136, 128, 266, 147],
    2020: [225, 273, -701, -20687, 2509, 4800, 1763, 1371, 661, 638, 245, -140],
    2021: [233, 468, 916, 278, 614, 962, 1091, 366, 312, 546, 249, 510],
    2022: [504, 714, 431, 428, 390, 372, 528, 315, 263, 261, 263, 223],
    2023: [517, 311, 236, 294, 339, 306, 187, 236, 297, 150, 199, 216],
    2024: [353, 275, 315, 165, 272, 206, 114, 142, 254, 227, 256, 212],
}

# CPI YoY (%)
CPI_DATA = {
    2019: [1.6, 1.5, 1.9, 2.0, 1.8, 1.6, 1.8, 1.7, 1.7, 1.8, 2.1, 2.3],
    2020: [2.5, 2.3, 1.5, 0.3, 0.1, 0.6, 1.0, 1.3, 1.4, 1.2, 1.2, 1.4],
    2021: [1.4, 1.7, 2.6, 4.2, 5.0, 5.4, 5.4, 5.3, 5.4, 6.2, 6.8, 7.0],
    2022: [7.5, 7.9, 8.5, 8.3, 8.6, 9.1, 8.5, 8.3, 8.2, 7.7, 7.1, 6.5],
    2023: [6.4, 6.0, 5.0, 4.9, 4.0, 3.0, 3.2, 3.7, 3.7, 3.2, 3.1, 3.4],
    2024: [3.1, 3.2, 3.5, 3.4, 3.3, 3.0, 2.9, 2.5, 2.4, 2.6, 2.7, 2.9],
}

# Fed Funds Rate (%)
FED_RATE_DATA = {
    2019: [2.50, 2.50, 2.50, 2.50, 2.50, 2.50, 2.25, 2.25, 2.00, 1.75, 1.75, 1.75],
    2020: [1.75, 1.75, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25],
    2021: [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25],
    2022: [0.25, 0.25, 0.50, 0.50, 1.00, 1.75, 2.50, 2.50, 3.25, 3.25, 4.00, 4.50],
    2023: [4.50, 4.75, 5.00, 5.00, 5.25, 5.25, 5.50, 5.50, 5.50, 5.50, 5.50, 5.50],
    2024: [5.50, 5.50, 5.50, 5.50, 5.50, 5.50, 5.50, 5.50, 5.00, 5.00, 4.75, 4.50],
}

# FOMC Meeting Dates
FOMC_DATES = {
    2019: ['01-30', '03-20', '05-01', '06-19', '07-31', '09-18', '10-30', '12-11'],
    2020: ['01-29', '03-03', '03-15', '04-29', '06-10', '07-29', '09-16', '11-05', '12-16'],
    2021: ['01-27', '03-17', '04-28', '06-16', '07-28', '09-22', '11-03', '12-15'],
    2022: ['01-26', '03-16', '05-04', '06-15', '07-27', '09-21', '11-02', '12-14'],
    2023: ['02-01', '03-22', '05-03', '06-14', '07-26', '09-20', '11-01', '12-13'],
    2024: ['01-31', '03-20', '05-01', '06-12', '07-31', '09-18', '11-07', '12-18'],
}

def get_first_friday(year, month):
    """Retourne le premier vendredi du mois."""
    first_day = datetime(year, month, 1)
    days_until_friday = (4 - first_day.weekday()) % 7
    return first_day + timedelta(days=days_until_friday)

def get_second_week_tuesday(year, month):
    """Retourne le mardi de la 2ème semaine."""
    first_day = datetime(year, month, 1)
    days_until_tuesday = (1 - first_day.weekday()) % 7
    first_tuesday = first_day + timedelta(days=days_until_tuesday)
    return first_tuesday + timedelta(days=7)

def create_economic_calendar(start_year=2019, end_year=2024):
    """Crée le calendar économique avec les vraies données."""
    events = []
    
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            # NFP - Premier vendredi
            nfp_date = get_first_friday(year, month)
            nfp_value = NFP_DATA.get(year, [200]*12)[month-1]
            nfp_prev = NFP_DATA.get(year, [200]*12)[month-2] if month > 1 else NFP_DATA.get(year-1, [200]*12)[-1]
            
            events.append({
                'Date': f"{nfp_date.strftime('%Y-%m-%d')} 12:30:00",
                'Event': 'Non-Farm Payrolls',
                'Impact': 'HIGH',
                'Actual': nfp_value,
                'Previous': nfp_prev,
                'Surprise': 'POSITIVE' if nfp_value > nfp_prev else 'NEGATIVE'
            })
            
            # CPI - 2ème semaine
            cpi_date = get_second_week_tuesday(year, month)
            cpi_value = CPI_DATA.get(year, [2.0]*12)[month-1]
            cpi_prev = CPI_DATA.get(year, [2.0]*12)[month-2] if month > 1 else CPI_DATA.get(year-1, [2.0]*12)[-1]
            
            events.append({
                'Date': f"{cpi_date.strftime('%Y-%m-%d')} 12:30:00",
                'Event': 'CPI y/y',
                'Impact': 'HIGH',
                'Actual': cpi_value,
                'Previous': cpi_prev,
                'Surprise': 'POSITIVE' if cpi_value > cpi_prev else 'NEGATIVE'
            })
        
        # FOMC Meetings
        for date_str in FOMC_DATES.get(year, []):
            try:
                month = int(date_str.split('-')[0])
                rate = FED_RATE_DATA.get(year, [2.0]*12)[month-1]
                prev_rate = FED_RATE_DATA.get(year, [2.0]*12)[month-2] if month > 1 else FED_RATE_DATA.get(year-1, [2.0]*12)[-1]
                
                events.append({
                    'Date': f"{year}-{date_str} 18:00:00",
                    'Event': 'FOMC Statement',
                    'Impact': 'HIGH',
                    'Actual': rate,
                    'Previous': prev_rate,
                    'Surprise': 'HAWKISH' if rate > prev_rate else ('DOVISH' if rate < prev_rate else 'NEUTRAL')
                })
            except:
                continue
    
    df = pd.DataFrame(events)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    return df

# Créer le calendar
calendar_df = create_economic_calendar(2019, 2024)
calendar_df.to_csv('data/economic_calendar_2019_2024.csv', index=False)

print(f"✅ Economic Calendar créé: {len(calendar_df)} événements")
print(f"\n📊 Par Event:")
print(calendar_df['Event'].value_counts())
print(f"\n📅 Aperçu:")
calendar_df.head(10)

## 3. Charger les Données Gold

In [ ]:
# Option 1: Uploader ton fichier Gold
from google.colab import files

print("📤 Upload ton fichier XAU_15MIN_2019_2024.csv")
uploaded = files.upload()

# Déplacer vers data/
for filename in uploaded.keys():
    os.rename(filename, f'data/{filename}')
    print(f"✅ Fichier uploadé: data/{filename}")

In [ ]:
# Charger et vérifier les données Gold
gold_df = pd.read_csv('data/XAU_15MIN_2019_2024.csv', parse_dates=['Date'])
gold_df = gold_df.set_index('Date')

print(f"📊 DONNÉES GOLD")
print(f"="*50)
print(f"Période: {gold_df.index.min()} à {gold_df.index.max()}")
print(f"Nombre de barres: {len(gold_df):,}")
print(f"Colonnes: {list(gold_df.columns)}")
print(f"\nAperçu:")
gold_df.head()

In [ ]:
# Visualiser les données
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Prix
axes[0].plot(gold_df.index, gold_df['Close'], linewidth=0.5)
axes[0].set_title('Gold Price (XAU/USD) 2019-2024')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(gold_df.index, gold_df['Volume'], width=0.01, alpha=0.5)
axes[1].set_title('Volume')
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Marquer les événements majeurs
print("\n📌 Événements majeurs dans les données:")
print("- Mars 2020: COVID Crash")
print("- Août 2020: Gold ATH $2075")
print("- 2022: Rate Hikes")
print("- Mars 2023: Banking Crisis")

## 4. Préparation des Données pour l'Entraînement

In [ ]:
# Split chronologique (IMPORTANT: pas de random split!)
total_len = len(gold_df)
train_end = int(total_len * 0.70)   # 70% train
val_end = int(total_len * 0.85)     # 15% validation

df_train = gold_df.iloc[:train_end].copy()
df_val = gold_df.iloc[train_end:val_end].copy()
df_test = gold_df.iloc[val_end:].copy()

print(f"📊 SPLIT DES DONNÉES")
print(f"="*50)
print(f"Train:      {len(df_train):>8,} barres ({df_train.index.min().date()} → {df_train.index.max().date()})")
print(f"Validation: {len(df_val):>8,} barres ({df_val.index.min().date()} → {df_val.index.max().date()})")
print(f"Test:       {len(df_test):>8,} barres ({df_test.index.min().date()} → {df_test.index.max().date()})")

# Sauvegarder les splits
df_train.to_csv('data/train.csv')
df_val.to_csv('data/validation.csv')
df_test.to_csv('data/test.csv')
print(f"\n✅ Splits sauvegardés dans data/")

In [ ]:
# Merger avec l'economic calendar
calendar_df = pd.read_csv('data/economic_calendar_2019_2024.csv', parse_dates=['Date'])

# Créer une colonne pour marquer les événements HIGH IMPACT
def mark_news_events(price_df, calendar_df, window_minutes=60):
    """
    Marque les barres qui sont proches d'un événement économique.
    
    Args:
        price_df: DataFrame avec les prix (index = Date)
        calendar_df: DataFrame avec les événements
        window_minutes: Fenêtre avant/après l'événement à marquer
    """
    price_df = price_df.copy()
    price_df['news_event'] = 0
    price_df['news_impact'] = 0.0
    price_df['news_type'] = ''
    
    for _, event in calendar_df.iterrows():
        event_time = event['Date']
        
        # Trouver les barres dans la fenêtre
        mask = (
            (price_df.index >= event_time - timedelta(minutes=window_minutes)) &
            (price_df.index <= event_time + timedelta(minutes=window_minutes))
        )
        
        price_df.loc[mask, 'news_event'] = 1
        price_df.loc[mask, 'news_impact'] = 1.0 if event['Impact'] == 'HIGH' else 0.5
        price_df.loc[mask, 'news_type'] = event['Event']
    
    return price_df

# Appliquer aux données
df_train_marked = mark_news_events(df_train, calendar_df)
df_val_marked = mark_news_events(df_val, calendar_df)
df_test_marked = mark_news_events(df_test, calendar_df)

print(f"📰 ÉVÉNEMENTS MARQUÉS")
print(f"="*50)
print(f"Train - Barres avec news: {df_train_marked['news_event'].sum():,} ({df_train_marked['news_event'].mean()*100:.1f}%)")
print(f"Val   - Barres avec news: {df_val_marked['news_event'].sum():,} ({df_val_marked['news_event'].mean()*100:.1f}%)")
print(f"Test  - Barres avec news: {df_test_marked['news_event'].sum():,} ({df_test_marked['news_event'].mean()*100:.1f}%)")

## 5. Création de l'Environnement de Trading

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from sklearn.preprocessing import StandardScaler
import ta

class TradingEnvironment(gym.Env):
    """
    Environnement de trading pour Gold avec intégration des news.
    
    Actions:
        0: HOLD
        1: BUY
        2: SELL
    
    Observations:
        - Features techniques (RSI, MACD, BB, etc.)
        - Position actuelle
        - Signaux news
    """
    
    def __init__(
        self,
        df: pd.DataFrame,
        initial_balance: float = 10000,
        max_position: float = 1.0,
        transaction_cost: float = 0.0002,
        use_news: bool = True
    ):
        super().__init__()
        
        self.df = df.copy()
        self.initial_balance = initial_balance
        self.max_position = max_position
        self.transaction_cost = transaction_cost
        self.use_news = use_news
        
        # Préparer les features
        self._prepare_features()
        
        # Espaces
        self.action_space = spaces.Discrete(3)  # HOLD, BUY, SELL
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.n_features,),
            dtype=np.float32
        )
        
        self.reset()
    
    def _prepare_features(self):
        """Prépare les features techniques."""
        df = self.df.copy()
        
        # Prix normalisés
        df['returns'] = df['Close'].pct_change()
        df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
        
        # RSI
        df['rsi'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()
        
        # MACD
        macd = ta.trend.MACD(df['Close'])
        df['macd'] = macd.macd()
        df['macd_signal'] = macd.macd_signal()
        df['macd_hist'] = macd.macd_diff()
        
        # Bollinger Bands
        bb = ta.volatility.BollingerBands(df['Close'])
        df['bb_high'] = bb.bollinger_hband()
        df['bb_low'] = bb.bollinger_lband()
        df['bb_pct'] = (df['Close'] - df['bb_low']) / (df['bb_high'] - df['bb_low'])
        
        # ATR
        df['atr'] = ta.volatility.AverageTrueRange(df['High'], df['Low'], df['Close']).average_true_range()
        
        # Moving Averages
        df['sma_20'] = df['Close'].rolling(20).mean()
        df['sma_50'] = df['Close'].rolling(50).mean()
        df['ema_12'] = df['Close'].ewm(span=12).mean()
        df['price_sma20_ratio'] = df['Close'] / df['sma_20']
        df['price_sma50_ratio'] = df['Close'] / df['sma_50']
        
        # Volatilité
        df['volatility'] = df['returns'].rolling(20).std() * np.sqrt(252 * 24 * 4)  # Annualisée pour 15min
        
        # Volume
        df['volume_sma'] = df['Volume'].rolling(20).mean()
        df['volume_ratio'] = df['Volume'] / df['volume_sma']
        
        # News features
        if self.use_news and 'news_event' in df.columns:
            df['news_signal'] = df['news_event']
            df['news_impact_signal'] = df['news_impact']
        else:
            df['news_signal'] = 0
            df['news_impact_signal'] = 0
        
        # Sélectionner les features
        self.feature_columns = [
            'returns', 'log_returns', 'rsi', 'macd', 'macd_signal', 'macd_hist',
            'bb_pct', 'atr', 'price_sma20_ratio', 'price_sma50_ratio',
            'volatility', 'volume_ratio', 'news_signal', 'news_impact_signal'
        ]
        
        # Supprimer les NaN
        df = df.dropna()
        
        # Normaliser
        self.scaler = StandardScaler()
        self.features = self.scaler.fit_transform(df[self.feature_columns])
        self.prices = df['Close'].values
        self.df_clean = df
        
        # +2 pour position et PnL
        self.n_features = len(self.feature_columns) + 2
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.current_step = 0
        self.balance = self.initial_balance
        self.position = 0  # -1, 0, 1
        self.entry_price = 0
        self.total_pnl = 0
        self.trades = []
        
        return self._get_observation(), {}
    
    def _get_observation(self):
        """Retourne l'observation actuelle."""
        features = self.features[self.current_step]
        position_feature = np.array([self.position, self.total_pnl / self.initial_balance])
        return np.concatenate([features, position_feature]).astype(np.float32)
    
    def step(self, action):
        """Exécute une action."""
        current_price = self.prices[self.current_step]
        reward = 0
        
        # Exécuter l'action
        if action == 1:  # BUY
            if self.position <= 0:
                # Fermer position short si existante
                if self.position < 0:
                    pnl = (self.entry_price - current_price) * abs(self.position)
                    pnl -= current_price * self.transaction_cost
                    self.total_pnl += pnl
                    reward += pnl / self.initial_balance
                
                # Ouvrir position long
                self.position = self.max_position
                self.entry_price = current_price
                self.balance -= current_price * self.transaction_cost
        
        elif action == 2:  # SELL
            if self.position >= 0:
                # Fermer position long si existante
                if self.position > 0:
                    pnl = (current_price - self.entry_price) * self.position
                    pnl -= current_price * self.transaction_cost
                    self.total_pnl += pnl
                    reward += pnl / self.initial_balance
                
                # Ouvrir position short
                self.position = -self.max_position
                self.entry_price = current_price
                self.balance -= current_price * self.transaction_cost
        
        # Avancer
        self.current_step += 1
        
        # Vérifier si terminé
        done = self.current_step >= len(self.prices) - 1
        
        # Fermer la position si terminé
        if done and self.position != 0:
            final_price = self.prices[self.current_step]
            if self.position > 0:
                pnl = (final_price - self.entry_price) * self.position
            else:
                pnl = (self.entry_price - final_price) * abs(self.position)
            self.total_pnl += pnl
            reward += pnl / self.initial_balance
        
        info = {
            'total_pnl': self.total_pnl,
            'position': self.position,
            'price': current_price
        }
        
        return self._get_observation(), reward, done, False, info

# Test de l'environnement
env = TradingEnvironment(df_train_marked, use_news=True)
obs, _ = env.reset()

print(f"✅ Environnement créé!")
print(f"   Observation shape: {obs.shape}")
print(f"   Action space: {env.action_space}")

## 6. Entraînement du Modèle PPO

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv
import torch

# Configuration de l'entraînement
TOTAL_TIMESTEPS = 500_000  # Ajuste selon ton temps disponible
EVAL_FREQ = 10_000

# Hyperparamètres optimisés pour trading
HYPERPARAMS = {
    'learning_rate': 3e-5,
    'n_steps': 2048,
    'batch_size': 128,
    'n_epochs': 10,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_range': 0.2,
    'ent_coef': 0.05,  # Exploration
    'vf_coef': 0.5,
    'max_grad_norm': 0.5,
}

print(f"🎯 Configuration de l'entraînement")
print(f"="*50)
print(f"Total timesteps: {TOTAL_TIMESTEPS:,}")
print(f"Hyperparamètres: {HYPERPARAMS}")

In [ ]:
# Callback pour logger les métriques
class TradingCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=10000, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.best_sharpe = -np.inf
        self.results = []
    
    def _on_step(self):
        if self.n_calls % self.eval_freq == 0:
            # Évaluer
            obs, _ = self.eval_env.reset()
            done = False
            returns = []
            
            while not done:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, done, _, info = self.eval_env.step(action)
                returns.append(reward)
            
            # Métriques
            returns = np.array(returns)
            sharpe = np.mean(returns) / (np.std(returns) + 1e-8) * np.sqrt(252 * 24 * 4)
            total_return = np.sum(returns)
            
            self.results.append({
                'timestep': self.n_calls,
                'sharpe': sharpe,
                'total_return': total_return
            })
            
            if sharpe > self.best_sharpe:
                self.best_sharpe = sharpe
                self.model.save('models/best_model')
            
            print(f"Step {self.n_calls:,}: Sharpe={sharpe:.2f}, Return={total_return:.2%}")
        
        return True

# Créer les environnements
train_env = TradingEnvironment(df_train_marked, use_news=True)
eval_env = TradingEnvironment(df_val_marked, use_news=True)

# Créer le callback
callback = TradingCallback(eval_env, eval_freq=EVAL_FREQ)

In [ ]:
# Créer et entraîner le modèle
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Device: {device}")

model = PPO(
    'MlpPolicy',
    train_env,
    verbose=1,
    tensorboard_log='logs/',
    device=device,
    **HYPERPARAMS
)

print(f"\n🚀 Démarrage de l'entraînement...")
print(f"="*50)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callback,
    progress_bar=True
)

# Sauvegarder le modèle final
model.save('models/final_model')
print(f"\n✅ Entraînement terminé!")
print(f"   Meilleur Sharpe: {callback.best_sharpe:.2f}")

## 7. Évaluation sur Données de Test

In [ ]:
# Charger le meilleur modèle
best_model = PPO.load('models/best_model')

# Évaluer sur test
test_env = TradingEnvironment(df_test_marked, use_news=True)
obs, _ = test_env.reset()

done = False
portfolio_values = [test_env.initial_balance]
actions_taken = []
positions = [0]

while not done:
    action, _ = best_model.predict(obs, deterministic=True)
    obs, reward, done, _, info = test_env.step(action)
    
    portfolio_values.append(test_env.initial_balance + test_env.total_pnl)
    actions_taken.append(action)
    positions.append(info['position'])

# Calculer les métriques
portfolio_values = np.array(portfolio_values)
returns = np.diff(portfolio_values) / portfolio_values[:-1]

# Sharpe Ratio
sharpe = np.mean(returns) / (np.std(returns) + 1e-8) * np.sqrt(252 * 24 * 4)

# Sortino Ratio
downside_returns = returns[returns < 0]
sortino = np.mean(returns) / (np.std(downside_returns) + 1e-8) * np.sqrt(252 * 24 * 4)

# Max Drawdown
peak = np.maximum.accumulate(portfolio_values)
drawdown = (peak - portfolio_values) / peak
max_dd = np.max(drawdown)

# Win Rate
win_rate = np.mean(returns > 0)

# Cumulative Return
cum_return = (portfolio_values[-1] / portfolio_values[0]) - 1

print(f"\n{'='*60}")
print(f"📊 RÉSULTATS SUR DONNÉES DE TEST")
print(f"{'='*60}")
print(f"")
print(f"  Sharpe Ratio:      {sharpe:>10.2f}")
print(f"  Sortino Ratio:     {sortino:>10.2f}")
print(f"  Max Drawdown:      {max_dd:>10.1%}")
print(f"  Win Rate:          {win_rate:>10.1%}")
print(f"  Cumulative Return: {cum_return:>10.1%}")
print(f"")
print(f"  Capital initial:   ${portfolio_values[0]:>10,.0f}")
print(f"  Capital final:     ${portfolio_values[-1]:>10,.0f}")
print(f"  Profit/Perte:      ${portfolio_values[-1]-portfolio_values[0]:>10,.0f}")
print(f"{'='*60}")

# Verdict
if sharpe >= 1.0 and max_dd <= 0.15:
    print(f"\n✅ MODÈLE PRÊT pour Paper Trading!")
else:
    print(f"\n⚠️ Modèle pas encore prêt - continue l'optimisation")

In [ ]:
# Visualisation des résultats
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# 1. Portfolio Value
axes[0].plot(portfolio_values, label='Portfolio', color='blue')
axes[0].axhline(y=test_env.initial_balance, color='gray', linestyle='--', label='Initial')
axes[0].fill_between(range(len(portfolio_values)), portfolio_values, test_env.initial_balance, 
                     where=portfolio_values >= test_env.initial_balance, alpha=0.3, color='green')
axes[0].fill_between(range(len(portfolio_values)), portfolio_values, test_env.initial_balance,
                     where=portfolio_values < test_env.initial_balance, alpha=0.3, color='red')
axes[0].set_title('Portfolio Value Over Time')
axes[0].set_ylabel('Value ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Drawdown
axes[1].fill_between(range(len(drawdown)), -drawdown * 100, 0, alpha=0.5, color='red')
axes[1].set_title(f'Drawdown (Max: {max_dd:.1%})')
axes[1].set_ylabel('Drawdown (%)')
axes[1].grid(True, alpha=0.3)

# 3. Actions Distribution
action_counts = pd.Series(actions_taken).value_counts().sort_index()
action_labels = ['HOLD', 'BUY', 'SELL']
colors = ['gray', 'green', 'red']
axes[2].bar([action_labels[i] for i in action_counts.index], action_counts.values, color=[colors[i] for i in action_counts.index])
axes[2].set_title('Actions Distribution')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 8. Télécharger le Modèle

In [ ]:
# Télécharger le meilleur modèle
from google.colab import files

# Compresser les modèles
!zip -r models.zip models/

# Télécharger
files.download('models.zip')
print("✅ Modèles téléchargés!")

## 9. Prochaines Étapes

1. **Si Sharpe > 1.0 et Max DD < 15%:**
   - Passer au Paper Trading Live
   - Connecter à MT5 en mode démo
   - Observer pendant 4+ semaines

2. **Si métriques insuffisantes:**
   - Augmenter `TOTAL_TIMESTEPS`
   - Ajuster les hyperparamètres
   - Ajouter plus de features
   - Essayer le curriculum learning